# Jacobik Conformer Ensemble Solver

This notebook demonstrates solving one of the predefined debug complexes from `interface/ligand_library.py` (`DEBUG_COMPLEX_CBTYCO`), resolving polyatomic ligands and binding modes from the library, and exporting ranked conformers.

In [ ]:
import importlib.util, sys
from pathlib import Path

repo = Path.cwd().resolve()
while repo != repo.parent and not (repo / "src").exists():
    repo = repo.parent
if importlib.util.find_spec("jacobik") is None and (repo / "src").is_dir():
    sys.path.insert(0, str(repo / "src"))
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import jacobik
from interface.ligand_library import (
    DEBUG_COMPLEX_CBTYCO,
    DEBUG_COMPLEX_OJIMIF,
    LIGANDS,
    ligand,
    ligands,
    list_ligands,
)

env_info = jacobik.check_environment()
print("Environment OK:", env_info)
print(f"Loaded ligand library ({len(LIGANDS)} polyatomic ligands)")


## 1. Inspect Constraints Database & Library Ligands

In [ ]:
db_path = jacobik.default_db_path()
print(f"Using constraints DB: {db_path}")
print("CBTYCO debug complex:", DEBUG_COMPLEX_CBTYCO.metal, DEBUG_COMPLEX_CBTYCO.geometry)
print("dmso modes:", list(LIGANDS["dmso"].modes.keys()))
print("Example O-bound dmso input:", ligand("dmso@kappa1_O"))


## 2. Solve `DEBUG_COMPLEX_CBTYCO` Directly

In [ ]:
result = jacobik.generate(
    *DEBUG_COMPLEX_CBTYCO,
    seed=11,
    n_conformers=3,
    device="cpu",
)
result.summary()


## 3. Build & Solve From Library Names / Binding Modes

In [ ]:
pt_mixed = jacobik.generate(
    metal="Pt",
    geometry="square_plan",
    charge=0,
    spin=1,
    ligands=["en", "SCN@kappa1_N", "dmso@kappa1_O"],
    seed=11,
    n_conformers=2,
    device="cpu",
)
pt_mixed.summary()


## 4. Export Top Conformers to XYZ

In [ ]:
out_dir = Path("/tmp/run_solver_output")
out_dir.mkdir(parents=True, exist_ok=True)
for c in result.conformers:
    xyz_path = out_dir / f"CBTYCO_conformer_{c.rank}.xyz"
    c.write_xyz(xyz_path)
    print(f"Saved rank {c.rank} (score: {c.score:.4f}) to {xyz_path}")

assert len(result.conformers) > 0
print("Export complete!")
